In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import chess
import chess.pgn
import math
import random
import os
import time
import pickle
from collections import deque
from torch.optim.lr_scheduler import StepLR
import mcts_engine
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from worker import AlphaZeroNet, batched_self_play, ResBlock

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Używam urządzenia: {device}")
print(dir(mcts_engine))

🚀 Używam urządzenia: cuda
['MCTSNode', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'batch_select_moves', 'mcts_engine']


In [2]:
def run_training(iterations=1000):
    try: mp.set_start_method('spawn', force=True)
    except: pass

    model = AlphaZeroNet().to(device)
    memory = deque(maxlen=20000) 
    num_workers = 4 
    
    if os.path.exists("chess_bot.pth"):
        model.load_state_dict(torch.load("chess_bot.pth", map_location=device))
        print("Model załadowany i sigma")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=500, gamma=0.9) 
    
    for i in range(iterations):
        print(f"--- Iteracja {i+1} ---")
        
        model.eval()
        
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            model_cpu = AlphaZeroNet()
            model_cpu.load_state_dict(model.state_dict())
            model_cpu = model_cpu.cpu()
            futures = [executor.submit(batched_self_play, model_cpu, num_games=5) for _ in range(num_workers)]
            
            for future in futures:
                try:
                    new_data = future.result()
                    memory.extend(new_data)
                except Exception as e:
                    import traceback
                    print(f"Exeption in worker: {e}")
                    print(traceback.format_exc())

        if len(memory) >= 2000:
            model.train()
            train_steps = 50
            memory_list = list(memory)
            for _ in range(train_steps):
                batch = random.sample(memory_list, 128)
                s_b = torch.from_numpy(np.stack([b[0] for b in batch])).to(device)
                p_b = torch.from_numpy(np.stack([b[1] for b in batch])).to(device)
                v_vals = np.array([b[2] for b in batch], dtype=np.float32)
                v_b = torch.from_numpy(v_vals).to(device)

                optimizer.zero_grad()
                p_out, v_out = model(s_b)
                
                log_probs = F.log_softmax(p_out, dim=1)
                
                p_loss = -torch.mean(torch.sum(p_b * log_probs, dim=1))
                
                v_loss = F.mse_loss(v_out.view(-1), v_b.view(-1))
                
                total_loss = p_loss + v_loss
                
                if torch.isnan(total_loss):
                    print("⚠️ NaN in loss")
                    continue

                total_loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
                
            scheduler.step()
            print(f"Iteracja {i+1} | Loss P: {p_loss:.4f} V: {v_loss:.4f} | Mem: {len(memory)}")
            
            print("💾 Zapisuję model...")
            torch.save(model.state_dict(), "chess_bot.pth")

In [ ]:
if __name__ == '__main__':
    run_training()

Model załadowany i sigma
--- Iteracja 1 ---
Iteracja 1 | Loss P: 4.2413 V: 0.0481 | Mem: 3804
💾 Zapisuję model...
--- Iteracja 2 ---
Iteracja 2 | Loss P: 4.3648 V: 0.0975 | Mem: 7248
💾 Zapisuję model...
--- Iteracja 3 ---
Iteracja 3 | Loss P: 4.2551 V: 0.0718 | Mem: 11524
💾 Zapisuję model...
--- Iteracja 4 ---
Iteracja 4 | Loss P: 3.8960 V: 0.1902 | Mem: 15463
💾 Zapisuję model...
--- Iteracja 5 ---
Iteracja 5 | Loss P: 4.0907 V: 0.1207 | Mem: 20000
💾 Zapisuję model...
--- Iteracja 6 ---
Iteracja 6 | Loss P: 3.9381 V: 0.1135 | Mem: 20000
💾 Zapisuję model...
--- Iteracja 7 ---
Iteracja 7 | Loss P: 3.7806 V: 0.1485 | Mem: 20000
💾 Zapisuję model...
--- Iteracja 8 ---
Iteracja 8 | Loss P: 4.0983 V: 0.0860 | Mem: 20000
💾 Zapisuję model...
--- Iteracja 9 ---
Iteracja 9 | Loss P: 3.7737 V: 0.1387 | Mem: 20000
💾 Zapisuję model...
--- Iteracja 10 ---
Iteracja 10 | Loss P: 3.6928 V: 0.1253 | Mem: 20000
💾 Zapisuję model...
--- Iteracja 11 ---
Iteracja 11 | Loss P: 3.7954 V: 0.1447 | Mem: 20000
💾 Z

In [ ]:
if os.path.exists("chess_bot.pth"):
    os.remove("chess_bot.pth")
    print("chess_bot.pth removed")

if 'memory' in locals():
    memory.clear()
    print("memory clear")